# 01b: Data Preprocessing

## Objective
Process weekly bed occupancy time series data from `01a_data_extraction.ipynb` and prepare it for forecasting models. Merge with external data sources (weather, flu) and create KPIs for downstream analysis.

## Tasks
- Load weekly bed occupancy data from `01a_data_extraction.ipynb`
- Validate weekly frequency and data quality
- Create KPIs: bed occupancy, occupancy rate, and additional metrics (ICU, ventilator, COVID patients) if available from extraction
- Merge external data: weather (Orlando, FL) and flu (Florida, US)
- Save processed weekly time series data for use in downstream notebooks

## Data Characteristics
**COVID-19 Hospital Capacity Dataset:**
- Data is already aggregated to weekly frequency (7-day averages)
- Dates are real calendar dates (not deidentified) - no transformation needed
- **Date Frequency Note**: According to CDC documentation, `collection_week` represents the starting Friday of the reporting period. However, the actual dates in the data appear as Sundays (possibly adjusted for reporting). The code uses `W-SUN` frequency to match the actual dates in the data.
- Bed occupancy represents average daily occupied beds over each 7-day period
- **ADVENTHEALTH ORLANDO quality data**: 2020-07-19 to 2024-04-21 (197 weeks)
- **Note**: Early weeks (2020-03-22 to 2020-07-12) were excluded in `01a_data_extraction.ipynb` due to missing occupancy data during early pandemic
- Hospital: ADVENTHEALTH ORLANDO, Orlando, FL

## Input Files
- `data/raw/occupancy_clean.csv` - Weekly bed occupancy from 01a
- `data/external/weather_orlando.csv` - Weekly weather data (optional)
- `data/external/flu_cdc.csv` - Weekly flu data (optional)

## Output Files
- `data/processed/weekly_occupancy.csv` - Weekly bed occupancy time series (all columns)
- `data/processed/weekly_kpis.csv` - Weekly KPIs (bed_occupancy, occupancy_rate, bed_capacity)
- `data/processed/weekly_occupancy_with_external.csv` - Combined with weather and flu features
- `data/processed/bed_occupancy.csv` - Simplified target variable file (bed_occupancy, occupancy_rate)
- `data/processed/preprocessing_summary.json` - Summary statistics and data quality metrics


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os
import json
from typing import Optional, Tuple, Dict

# Setup paths - Google Drive (for Colab)
DRIVE_ROOT = None
DRIVE_DATA_RAW = None
DRIVE_DATA_PROCESSED = None
DRIVE_DATA_EXTERNAL = None

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hospital_occupancy_forecasting'
    DRIVE_DATA_RAW = os.path.join(DRIVE_ROOT, 'data', 'raw')
    DRIVE_DATA_PROCESSED = os.path.join(DRIVE_ROOT, 'data', 'processed')
    DRIVE_DATA_EXTERNAL = os.path.join(DRIVE_ROOT, 'data', 'external')
    for p in [DRIVE_DATA_RAW, DRIVE_DATA_PROCESSED, DRIVE_DATA_EXTERNAL]:
        os.makedirs(p, exist_ok=True)
except:
    pass

# Detect environment (Colab vs local)
IS_COLAB = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# Setup paths - works for both Colab and local environments
if IS_COLAB:
    PROJECT_ROOT = '/content/hospital_occupancy_forecasting'
else:
    # Local environment: use current working directory
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
    if not os.path.exists(PROJECT_ROOT) or 'hospital_occupancy_forecasting' not in PROJECT_ROOT:
        # Fallback: assume we're in the project root
        PROJECT_ROOT = os.path.abspath('.')

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
DATA_RAW = os.path.join(DATA_DIR, 'raw')
DATA_PROCESSED = os.path.join(DATA_DIR, 'processed')
DATA_EXTERNAL = os.path.join(DATA_DIR, 'external')

for p in [DATA_RAW, DATA_PROCESSED, DATA_EXTERNAL]:
    os.makedirs(p, exist_ok=True)

print(f"✓ Setup complete")
print(f"  Environment: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"  Project root: {PROJECT_ROOT}")
print(f"  Data (raw): {DATA_RAW}")
print(f"  Data (processed): {DATA_PROCESSED}")
print(f"  Data (external): {DATA_EXTERNAL}")

Mounted at /content/drive
✓ Setup complete
  Environment: Google Colab
  Project root: /content/hospital_occupancy_forecasting
  Data (raw): /content/hospital_occupancy_forecasting/data/raw
  Data (processed): /content/hospital_occupancy_forecasting/data/processed
  Data (external): /content/hospital_occupancy_forecasting/data/external


## Helper Functions

Simple helper functions for loading and validating weekly bed occupancy data.


In [ ]:
# ============================================================================
# Helper Functions for Weekly Data Processing
# ============================================================================

def load_weekly_time_series(file_path: str) -> pd.DataFrame:
    """Load and validate weekly bed occupancy time series data.

    Args:
        file_path: Path to occupancy_clean.csv file

    Returns:
        DataFrame with weekly bed occupancy data (datetime index)

    Raises:
        FileNotFoundError: If file doesn't exist
        ValueError: If required columns are missing or data is invalid
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    # Load data
    df = pd.read_csv(file_path)

    if df.empty:
        raise ValueError("Loaded DataFrame is empty")

    # Parse date columns (try multiple possible column names)
    date_cols = ['aligned_date', 'date']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    # Validate required columns
    required_cols = ['bed_occupancy']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Determine which date column to use as index
    index_col = None
    for col in ['date', 'aligned_date']:
        if col in df.columns and df[col].notna().any():
            index_col = col
            break

    if index_col is None:
        raise ValueError("No valid date column found (expected 'date' or 'aligned_date')")

    # Set date as index
    df = df.set_index(index_col).sort_index()

    # DIAGNOSTIC: Verify day of week
    if len(df) > 0:
        first_date = df.index[0]
        last_date = df.index[-1]
        print(f"\n  📊 HOSPITAL DATA DIAGNOSTICS:")
        print(f"     Index column used: {index_col}")
        print(f"     First date: {first_date.date()} ({first_date.strftime('%A')})")
        print(f"     Last date: {last_date.date()} ({last_date.strftime('%A')})")

        # Check if all dates are the same day of week
        days_of_week = df.index.dayofweek
        unique_days = days_of_week.unique()
        if len(unique_days) == 1:
            day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
            print(f"     ✓ All dates are {day_names[unique_days[0]]}s")
        else:
            print(f"     ⚠️  Mixed days of week found: {len(unique_days)} different days")

    return df


def validate_weekly_frequency(df: pd.DataFrame) -> Tuple[bool, int]:
    """Validate that data has weekly frequency (7-day intervals).

    Args:
        df: DataFrame with datetime index

    Returns:
        Tuple of (is_valid, num_invalid_intervals)
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame index must be DatetimeIndex")

    if len(df) < 2:
        return True, 0  # Single record, can't validate frequency

    # Calculate differences between consecutive dates
    date_diffs = pd.Series(df.index).diff().dt.days.dropna()

    # Check if differences are approximately 7 days (allow 6-8 day range for week boundaries)
    valid_diffs = date_diffs[(date_diffs >= 6) & (date_diffs <= 8)]
    invalid_count = len(date_diffs) - len(valid_diffs)

    is_valid = invalid_count == 0 or (invalid_count / len(date_diffs)) < 0.1  # Allow 10% tolerance

    return is_valid, invalid_count


def create_kpi_dataframe(weekly_data: pd.DataFrame) -> pd.DataFrame:
    """Create a DataFrame with KPIs for weekly bed occupancy data.

    Args:
        weekly_data: DataFrame with weekly bed occupancy data

    Returns:
        DataFrame with KPIs including:
        - Core: bed_occupancy, occupancy_rate, bed_capacity
        - ICU: icu_occupied, icu_capacity, icu_occupancy_rate (if available)
        - Ventilator: ventilator_used, ventilator_available, ventilator_utilization_rate (if available)
        - COVID: covid_inpatients, covid_icu, covid_patient_pct (if available)
        - Other: staffed_beds (if available)
    """
    if weekly_data.empty:
        raise ValueError("weekly_data is empty. Cannot create KPI DataFrame.")

    kpi_cols = {}

    # Primary KPIs: bed occupancy
    if 'bed_occupancy' in weekly_data.columns:
        kpi_cols['bed_occupancy'] = weekly_data['bed_occupancy']

    if 'occupancy_rate' in weekly_data.columns:
        kpi_cols['occupancy_rate'] = weekly_data['occupancy_rate']
    elif 'bed_occupancy' in weekly_data.columns and 'bed_capacity' in weekly_data.columns:
        # Calculate occupancy rate if not present
        kpi_cols['occupancy_rate'] = (weekly_data['bed_occupancy'] / weekly_data['bed_capacity']) * 100

    if 'bed_capacity' in weekly_data.columns:
        kpi_cols['bed_capacity'] = weekly_data['bed_capacity']

    # ICU KPIs (if available)
    if 'icu_occupied' in weekly_data.columns:
        kpi_cols['icu_occupied'] = weekly_data['icu_occupied']

    if 'icu_capacity' in weekly_data.columns:
        kpi_cols['icu_capacity'] = weekly_data['icu_capacity']

    if 'icu_occupancy_rate' in weekly_data.columns:
        kpi_cols['icu_occupancy_rate'] = weekly_data['icu_occupancy_rate']
    elif 'icu_occupied' in weekly_data.columns and 'icu_capacity' in weekly_data.columns:
        # Calculate ICU occupancy rate if not present
        mask = (weekly_data['icu_capacity'].notna()) & (weekly_data['icu_capacity'] > 0)
        icu_rate = pd.Series(index=weekly_data.index, dtype=float)
        icu_rate.loc[mask] = (
            weekly_data.loc[mask, 'icu_occupied'] / weekly_data.loc[mask, 'icu_capacity']
        ) * 100
        kpi_cols['icu_occupancy_rate'] = icu_rate

    # Ventilator KPIs (if available)
    if 'ventilator_used' in weekly_data.columns:
        kpi_cols['ventilator_used'] = weekly_data['ventilator_used']

    if 'ventilator_available' in weekly_data.columns:
        kpi_cols['ventilator_available'] = weekly_data['ventilator_available']

    if 'ventilator_utilization_rate' in weekly_data.columns:
        kpi_cols['ventilator_utilization_rate'] = weekly_data['ventilator_utilization_rate']
    elif 'ventilator_used' in weekly_data.columns and 'ventilator_available' in weekly_data.columns:
        # Calculate ventilator utilization rate if not present
        mask = (weekly_data['ventilator_available'].notna()) & (weekly_data['ventilator_available'] > 0)
        vent_rate = pd.Series(index=weekly_data.index, dtype=float)
        vent_rate.loc[mask] = (
            weekly_data.loc[mask, 'ventilator_used'] / weekly_data.loc[mask, 'ventilator_available']
        ) * 100
        kpi_cols['ventilator_utilization_rate'] = vent_rate

    # COVID patient KPIs (if available)
    if 'covid_inpatients' in weekly_data.columns:
        kpi_cols['covid_inpatients'] = weekly_data['covid_inpatients']

    if 'covid_icu' in weekly_data.columns:
        kpi_cols['covid_icu'] = weekly_data['covid_icu']

    if 'covid_patient_pct' in weekly_data.columns:
        kpi_cols['covid_patient_pct'] = weekly_data['covid_patient_pct']
    elif 'covid_inpatients' in weekly_data.columns and 'bed_occupancy' in weekly_data.columns:
        # Calculate COVID patient percentage if not present
        mask = (weekly_data['bed_occupancy'].notna()) & (weekly_data['bed_occupancy'] > 0)
        covid_pct = pd.Series(index=weekly_data.index, dtype=float)
        covid_pct.loc[mask] = (
            weekly_data.loc[mask, 'covid_inpatients'] / weekly_data.loc[mask, 'bed_occupancy']
        ) * 100
        kpi_cols['covid_patient_pct'] = covid_pct

    # Staffed beds (if available)
    if 'staffed_beds' in weekly_data.columns:
        kpi_cols['staffed_beds'] = weekly_data['staffed_beds']

    if not kpi_cols:
        raise ValueError("No KPI columns found. Expected at least 'bed_occupancy'")

    kpi_df = pd.DataFrame(kpi_cols)
    return kpi_df


def load_external_data(external_data_dir: str) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    """Load external data (weather, flu) from files.

    Note: These files should be pre-aligned to hospital dates by 01a_data_extraction.ipynb.
    The data is expected to have dates matching the hospital data exactly.

    Args:
        external_data_dir: Path to data/external/ directory

    Returns:
        tuple: (weather_df, flu_df) - DataFrames with datetime index, ready to merge, or None if not available
    """
    weather_df = None
    flu_df = None

    # Load weather data (Orlando, FL) - should be pre-aligned in 01a
    weather_file = os.path.join(external_data_dir, 'weather_orlando.csv')
    if os.path.exists(weather_file):
        try:
            weather_df = pd.read_csv(weather_file, parse_dates=['date'])
            weather_df = weather_df.set_index('date').sort_index()
            print(f"  ✓ Loaded weather data: {len(weather_df)} records")
            print(f"    Date range: {weather_df.index.min().date()} to {weather_df.index.max().date()}")

            # DIAGNOSTIC: Check day of week
            first_date = weather_df.index[0]
            print(f"    First date: {first_date.date()} ({first_date.strftime('%A')})")
            if first_date.strftime('%A') != 'Sunday':
                print(f"    ⚠️  WARNING: Weather dates are NOT Sundays! May cause merge issues.")

            # DIAGNOSTIC: Check for non-null data
            numeric_cols = weather_df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) > 0:
                non_null_pct = weather_df[numeric_cols].notna().sum().sum() / (len(weather_df) * len(numeric_cols)) * 100
                print(f"    Data completeness: {non_null_pct:.1f}% non-null values")
        except Exception as e:
            print(f"  ⚠️  Error loading weather data: {e}")
    else:
        print(f"  ⚠️  Weather file not found: {weather_file}")

    # Load flu data (Florida, US) - should be pre-aligned in 01a
    flu_file = os.path.join(external_data_dir, 'flu_cdc.csv')
    if os.path.exists(flu_file):
        try:
            flu_df = pd.read_csv(flu_file, parse_dates=['date'])
            flu_df = flu_df.set_index('date').sort_index()
            print(f"  ✓ Loaded flu data: {len(flu_df)} records")
            print(f"    Date range: {flu_df.index.min().date()} to {flu_df.index.max().date()}")

            # DIAGNOSTIC: Check day of week
            first_date = flu_df.index[0]
            print(f"    First date: {first_date.date()} ({first_date.strftime('%A')})")
            if first_date.strftime('%A') != 'Sunday':
                print(f"    ⚠️  WARNING: Flu dates are NOT Sundays! May cause merge issues.")

            # DIAGNOSTIC: Check for flu_activity_level
            if 'flu_activity_level' in flu_df.columns:
                non_null = flu_df['flu_activity_level'].notna().sum()
                print(f"    flu_activity_level: {non_null}/{len(flu_df)} non-null ({non_null/len(flu_df)*100:.1f}%)")
        except Exception as e:
            print(f"  ⚠️  Error loading flu data: {e}")
    else:
        print(f"  ⚠️  Flu file not found: {flu_file}")

    return weather_df, flu_df


def merge_external_data(
    weekly_data: pd.DataFrame,
    weather_df: Optional[pd.DataFrame],
    flu_df: Optional[pd.DataFrame]
) -> pd.DataFrame:
    """Merge external data (weather, flu) with weekly bed occupancy data.

    This function attempts exact date matching first, then falls back to
    nearest-date matching within 7 days if exact matches fail.

    Args:
        weekly_data: DataFrame with weekly bed occupancy data (datetime index)
        weather_df: Weather DataFrame with datetime index (or None)
        flu_df: Flu DataFrame with datetime index (or None)

    Returns:
        DataFrame with merged data
    """
    merged = weekly_data.copy()

    print(f"\n  📊 MERGE DIAGNOSTICS:")
    print(f"     Hospital data: {len(weekly_data)} records")
    print(f"     Hospital date range: {weekly_data.index.min().date()} to {weekly_data.index.max().date()}")
    print(f"     Hospital first date day: {weekly_data.index[0].strftime('%A')}")

    # Merge weather data
    if weather_df is not None and len(weather_df) > 0:
        weather_cols = ['temp_avg', 'temp_max', 'temp_min', 'precipitation', 'wind_speed', 'pressure']
        weather_cols = [col for col in weather_cols if col in weather_df.columns]

        if weather_cols:
            print(f"\n  🌤️  Weather Merge:")
            print(f"     Weather records: {len(weather_df)}")
            print(f"     Weather date range: {weather_df.index.min().date()} to {weather_df.index.max().date()}")
            print(f"     Weather first date day: {weather_df.index[0].strftime('%A')}")

            # Check for exact date matches
            weekly_dates_set = set(weekly_data.index)
            weather_dates_set = set(weather_df.index)
            exact_matches = len(weekly_dates_set & weather_dates_set)

            print(f"     Exact date matches: {exact_matches}/{len(weekly_data)} ({exact_matches/len(weekly_data)*100:.1f}%)")

            if exact_matches == len(weekly_data):
                # Perfect match - use simple join
                merged = merged.join(weather_df[weather_cols], how='left')
                print(f"     ✓ Perfect alignment - used direct join")
            elif exact_matches > len(weekly_data) * 0.9:
                # Good match - use simple join
                merged = merged.join(weather_df[weather_cols], how='left')
                print(f"     ✓ Good alignment - used direct join ({exact_matches}/{len(weekly_data)} exact matches)")
            else:
                # Need nearest-date matching
                print(f"     ⚠️  Poor alignment - using nearest-date matching (within 7 days)")

                # Reindex and fill with nearest matches
                weather_aligned = weather_df[weather_cols].reindex(weekly_data.index)

                # Fill missing dates with nearest match within 7 days
                missing_mask = weather_aligned[weather_cols[0]].isna()
                filled_count = 0
                for idx in weekly_data.index[missing_mask]:
                    date_diffs = abs(weather_df.index - idx)
                    min_diff = date_diffs.min()
                    if min_diff <= pd.Timedelta(days=7):
                        nearest_idx = weather_df.index[date_diffs.argmin()]
                        weather_aligned.loc[idx] = weather_df.loc[nearest_idx, weather_cols]
                        filled_count += 1

                print(f"     Filled {filled_count} dates using nearest match")
                merged = merged.join(weather_aligned, how='left')

            # Check for missing values after merge
            missing_weather = merged[weather_cols].isnull().sum().sum()
            total_values = len(merged) * len(weather_cols)
            if missing_weather > 0:
                print(f"     ⚠️  Missing after merge: {missing_weather}/{total_values} ({missing_weather/total_values*100:.1f}%)")
                for col in weather_cols:
                    col_missing = merged[col].isnull().sum()
                    if col_missing > 0:
                        print(f"        {col}: {col_missing} missing ({col_missing/len(merged)*100:.1f}%)")
            else:
                print(f"     ✓ All weather values present after merge")

    # Merge flu data
    if flu_df is not None and len(flu_df) > 0:
        flu_cols = ['flu_activity_level', 'wili', 'ili', 'is_flu_season', 'is_proxy']
        flu_cols = [col for col in flu_cols if col in flu_df.columns]

        if flu_cols:
            print(f"\n  🦠 Flu Merge:")
            print(f"     Flu records: {len(flu_df)}")
            print(f"     Flu date range: {flu_df.index.min().date()} to {flu_df.index.max().date()}")
            print(f"     Flu first date day: {flu_df.index[0].strftime('%A')}")

            # Check for exact date matches
            weekly_dates_set = set(weekly_data.index)
            flu_dates_set = set(flu_df.index)
            exact_matches = len(weekly_dates_set & flu_dates_set)

            print(f"     Exact date matches: {exact_matches}/{len(weekly_data)} ({exact_matches/len(weekly_data)*100:.1f}%)")

            if exact_matches == len(weekly_data):
                # Perfect match - use simple join
                merged = merged.join(flu_df[flu_cols], how='left')
                print(f"     ✓ Perfect alignment - used direct join")
            elif exact_matches > len(weekly_data) * 0.9:
                # Good match - use simple join
                merged = merged.join(flu_df[flu_cols], how='left')
                print(f"     ✓ Good alignment - used direct join ({exact_matches}/{len(weekly_data)} exact matches)")
            else:
                # Need nearest-date matching
                print(f"     ⚠️  Poor alignment - using nearest-date matching (within 7 days)")

                # Reindex and fill with nearest matches
                flu_aligned = flu_df[flu_cols].reindex(weekly_data.index)

                # Fill missing dates with nearest match within 7 days
                missing_mask = flu_aligned[flu_cols[0]].isna()
                filled_count = 0
                for idx in weekly_data.index[missing_mask]:
                    date_diffs = abs(flu_df.index - idx)
                    min_diff = date_diffs.min()
                    if min_diff <= pd.Timedelta(days=7):
                        nearest_idx = flu_df.index[date_diffs.argmin()]
                        flu_aligned.loc[idx] = flu_df.loc[nearest_idx, flu_cols]
                        filled_count += 1

                print(f"     Filled {filled_count} dates using nearest match")
                merged = merged.join(flu_aligned, how='left')

            # Check for missing values after merge
            missing_flu = merged[flu_cols].isnull().sum().sum()
            total_values = len(merged) * len(flu_cols)
            if missing_flu > 0:
                print(f"     ⚠️  Missing after merge: {missing_flu}/{total_values} ({missing_flu/total_values*100:.1f}%)")
                for col in flu_cols:
                    col_missing = merged[col].isnull().sum()
                    if col_missing > 0:
                        print(f"        {col}: {col_missing} missing ({col_missing/len(merged)*100:.1f}%)")
            else:
                print(f"     ✓ All flu values present after merge")

    return merged


print("✓ Helper functions defined")

✓ Helper functions defined


## Load Weekly Bed Occupancy Data

Load the cleaned weekly bed occupancy data from `01a_data_extraction.ipynb`.


In [ ]:
# Load cleaned occupancy data from 01a_data_extraction.ipynb
print("="*60)
print("LOADING WEEKLY BED OCCUPANCY DATA")
print("="*60)

# Search for the data file in multiple locations
file_paths = [
    os.path.join(DRIVE_DATA_RAW, 'occupancy_clean.csv') if DRIVE_DATA_RAW else None,
    os.path.join(DATA_RAW, 'occupancy_clean.csv'),
]
file_paths = [p for p in file_paths if p]  # Remove None values

weekly_data = None
loaded_file = None

for file_path in file_paths:
    if os.path.exists(file_path):
        try:
            weekly_data = load_weekly_time_series(file_path)
            loaded_file = file_path
            print(f"\n✓ Loaded from: {file_path}")
            break
        except Exception as e:
            print(f"⚠️  Error loading {file_path}: {e}")
            continue

if weekly_data is None:
    raise FileNotFoundError(
        f"❌ Data file not found. Run notebook 01a_data_extraction.ipynb first.\n"
        f"   Searched in: {file_paths}"
    )

# Display data info
print(f"\n📊 Data Overview:")
print(f"  Shape: {weekly_data.shape[0]} weeks × {weekly_data.shape[1]} columns")
print(f"  Date range: {weekly_data.index.min().date()} to {weekly_data.index.max().date()}")
print(f"  Columns: {list(weekly_data.columns)}")

# Check for key columns
print(f"\n📋 Key Columns:")
key_cols = ['bed_occupancy', 'bed_capacity', 'occupancy_rate', 'is_imputed']
# Additional metrics from 01a (if available)
additional_cols = ['icu_occupied', 'icu_capacity', 'icu_occupancy_rate',
                   'ventilator_used', 'ventilator_available', 'ventilator_utilization_rate',
                   'covid_inpatients', 'covid_icu', 'covid_patient_pct', 'staffed_beds']
all_key_cols = key_cols + additional_cols

for col in all_key_cols:
    if col in weekly_data.columns:
        non_null = weekly_data[col].notna().sum()
        print(f"  ✓ {col}: {non_null} non-null ({non_null/len(weekly_data)*100:.1f}%)")
    elif col in key_cols:
        print(f"  ⚠️ {col}: not present")

# Show sample data
print(f"\n📋 Sample Data (first 5 rows):")
display_cols = ['bed_occupancy', 'bed_capacity', 'occupancy_rate', 'is_imputed']
# Include additional metrics if available
additional_display_cols = ['icu_occupied', 'icu_capacity', 'ventilator_used', 'covid_inpatients']
for col in additional_display_cols:
    if col in weekly_data.columns:
        display_cols.append(col)
display_cols = [col for col in display_cols if col in weekly_data.columns]
print(weekly_data[display_cols].head().to_string())

LOADING WEEKLY BED OCCUPANCY DATA

  📊 HOSPITAL DATA DIAGNOSTICS:
     Index column used: date
     First date: 2020-07-19 (Sunday)
     Last date: 2024-04-21 (Sunday)
     ✓ All dates are Sundays

✓ Loaded from: /content/drive/MyDrive/hospital_occupancy_forecasting/data/raw/occupancy_clean.csv

📊 Data Overview:
  Shape: 197 weeks × 16 columns
  Date range: 2020-07-19 to 2024-04-21
  Columns: ['aligned_date', 'bed_occupancy', 'bed_capacity', 'occupancy_rate', 'coverage', 'hospital_name', 'city', 'state', 'is_imputed', 'icu_occupied', 'icu_capacity', 'covid_inpatients', 'covid_icu', 'staffed_beds', 'icu_occupancy_rate', 'covid_patient_pct']

📋 Key Columns:
  ✓ bed_occupancy: 197 non-null (100.0%)
  ✓ bed_capacity: 197 non-null (100.0%)
  ✓ occupancy_rate: 197 non-null (100.0%)
  ✓ is_imputed: 197 non-null (100.0%)
  ✓ icu_occupied: 195 non-null (99.0%)
  ✓ icu_capacity: 195 non-null (99.0%)
  ✓ icu_occupancy_rate: 195 non-null (99.0%)
  ✓ covid_inpatients: 151 non-null (76.6%)
  ✓ covid

## Validate Weekly Frequency

Verify that the data has weekly frequency (approximately 7 days between records).


In [ ]:
# Validate weekly frequency
print("="*60)
print("VALIDATING WEEKLY FREQUENCY")
print("="*60)

is_valid, invalid_count = validate_weekly_frequency(weekly_data)

print(f"\n📊 Frequency Validation:")
print(f"  Total records: {len(weekly_data)}")
print(f"  Date range: {weekly_data.index.min().date()} to {weekly_data.index.max().date()}")

# Calculate expected vs actual weeks
date_range_days = (weekly_data.index.max() - weekly_data.index.min()).days
expected_weeks = date_range_days // 7 + 1
print(f"  Expected weeks (based on date range): ~{expected_weeks}")
print(f"  Actual weeks: {len(weekly_data)}")

if is_valid:
    print(f"\n✓ Weekly frequency validated successfully")
    if invalid_count > 0:
        print(f"  Note: {invalid_count} intervals slightly differ from 7 days (within tolerance)")
else:
    print(f"\n⚠️  Warning: {invalid_count} date intervals are not ~7 days")
    print(f"   This may indicate missing weeks or data quality issues")

# Check for missing weeks
if len(weekly_data) < expected_weeks * 0.9:  # More than 10% missing
    missing_pct = (1 - len(weekly_data) / expected_weeks) * 100
    print(f"\n⚠️  Warning: ~{missing_pct:.1f}% of expected weeks may be missing")
else:
    print(f"\n✓ Data coverage looks complete")

VALIDATING WEEKLY FREQUENCY

📊 Frequency Validation:
  Total records: 197
  Date range: 2020-07-19 to 2024-04-21
  Expected weeks (based on date range): ~197
  Actual weeks: 197

✓ Weekly frequency validated successfully

✓ Data coverage looks complete


## Data Quality Checks

Perform quality checks on the weekly bed occupancy data.


In [ ]:
# Data quality checks
print("="*60)
print("DATA QUALITY CHECKS")
print("="*60)

# 1. Missing values check
print(f"\n📊 Missing Values:")
missing_counts = weekly_data.isnull().sum()
if missing_counts.sum() == 0:
    print(f"  ✓ No missing values in any column")
else:
    for col, count in missing_counts[missing_counts > 0].items():
        pct = count / len(weekly_data) * 100
        print(f"  ⚠️  {col}: {count} missing ({pct:.1f}%)")

# 2. Bed occupancy validation
if 'bed_occupancy' in weekly_data.columns:
    print(f"\n📊 Bed Occupancy Validation:")
    occ = weekly_data['bed_occupancy'].dropna()

    # Check for negative values
    negative_count = (occ < 0).sum()
    if negative_count > 0:
        print(f"  ⚠️  Found {negative_count} negative values (unrealistic)")
    else:
        print(f"  ✓ No negative values")

    # Check for outliers (using IQR method)
    q1, q3 = occ.quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = ((occ < q1 - 1.5 * iqr) | (occ > q3 + 1.5 * iqr)).sum()
    if outliers > 0:
        print(f"  ℹ️  Found {outliers} potential outliers (IQR method)")
    else:
        print(f"  ✓ No extreme outliers detected")

    # Statistics
    print(f"\n  Statistics:")
    print(f"    Mean: {occ.mean():.1f} beds")
    print(f"    Median: {occ.median():.1f} beds")
    print(f"    Std: {occ.std():.1f} beds")
    print(f"    Min: {occ.min():.1f} beds")
    print(f"    Max: {occ.max():.1f} beds")

# 3. Occupancy rate validation
if 'occupancy_rate' in weekly_data.columns:
    print(f"\n📊 Occupancy Rate Validation:")
    rate = weekly_data['occupancy_rate'].dropna()

    # Check for values > 100%
    over_100 = (rate > 100).sum()
    if over_100 > 0:
        print(f"  ⚠️  Found {over_100} records with >100% occupancy")
        print(f"     (May indicate surge capacity or data issues)")
    else:
        print(f"  ✓ All values ≤100%")

    print(f"\n  Statistics:")
    print(f"    Mean: {rate.mean():.1f}%")
    print(f"    Median: {rate.median():.1f}%")
    print(f"    Min: {rate.min():.1f}%")
    print(f"    Max: {rate.max():.1f}%")

# 4. Imputed weeks check
if 'is_imputed' in weekly_data.columns:
    print(f"\n📊 Imputed Weeks:")
    imputed_count = weekly_data['is_imputed'].sum()
    if imputed_count > 0:
        pct = imputed_count / len(weekly_data) * 100
        print(f"  ℹ️  {imputed_count} weeks were imputed ({pct:.1f}%)")
        print(f"     (Imputed via time-aware linear interpolation in 01a)")
    else:
        print(f"  ✓ No imputed weeks (complete data)")

print(f"\n✅ Data quality checks complete")

DATA QUALITY CHECKS

📊 Missing Values:
  ⚠️  icu_occupied: 2 missing (1.0%)
  ⚠️  icu_capacity: 2 missing (1.0%)
  ⚠️  covid_inpatients: 46 missing (23.4%)
  ⚠️  covid_icu: 8 missing (4.1%)
  ⚠️  icu_occupancy_rate: 2 missing (1.0%)
  ⚠️  covid_patient_pct: 46 missing (23.4%)

📊 Bed Occupancy Validation:
  ✓ No negative values
  ℹ️  Found 8 potential outliers (IQR method)

  Statistics:
    Mean: 2254.4 beds
    Median: 2269.4 beds
    Std: 136.5 beds
    Min: 1882.0 beds
    Max: 2510.7 beds

📊 Occupancy Rate Validation:
  ✓ All values ≤100%

  Statistics:
    Mean: 90.1%
    Median: 92.0%
    Min: 66.8%
    Max: 98.4%

📊 Imputed Weeks:
  ✓ No imputed weeks (complete data)

✅ Data quality checks complete


## Create KPI DataFrame

Create a KPI DataFrame with the key metrics for forecasting.


In [ ]:
# Create KPI DataFrame
print("="*60)
print("CREATING KPI DATAFRAME")
print("="*60)

kpi_df = create_kpi_dataframe(weekly_data)

print(f"\n✓ KPI DataFrame created:")
print(f"  Shape: {kpi_df.shape}")
print(f"  Columns: {list(kpi_df.columns)}")
print(f"  Date range: {kpi_df.index.min().date()} to {kpi_df.index.max().date()}")

# Show summary statistics
print(f"\n📊 KPI Summary Statistics:")
print(kpi_df.describe().round(2).to_string())

# Show sample
print(f"\n📋 Sample KPI Data (first 5 rows):")
print(kpi_df.head().to_string())

CREATING KPI DATAFRAME

✓ KPI DataFrame created:
  Shape: (197, 10)
  Columns: ['bed_occupancy', 'occupancy_rate', 'bed_capacity', 'icu_occupied', 'icu_capacity', 'icu_occupancy_rate', 'covid_inpatients', 'covid_icu', 'covid_patient_pct', 'staffed_beds']
  Date range: 2020-07-19 to 2024-04-21

📊 KPI Summary Statistics:
       bed_occupancy  occupancy_rate  bed_capacity  icu_occupied  icu_capacity  icu_occupancy_rate  covid_inpatients  covid_icu  covid_patient_pct  staffed_beds
count         197.00          197.00        197.00        195.00        195.00              195.00            151.00     189.00             151.00        197.00
mean         2254.37           90.14       2509.68        276.82        341.18               81.69            201.25      29.97               9.14       2509.68
std           136.47            6.93        164.43         21.99         25.75                9.81            168.16      34.33               7.60        164.43
min          1882.00           66.8

In [ ]:
# ============================================================================
# DIAGNOSTIC: Check hospital data after loading
# ============================================================================

print("="*60)
print("DIAGNOSTIC: HOSPITAL DATA CHECK")
print("="*60)

# Check weekly_data after loading
print(f"\n📊 Hospital Weekly Data:")
print(f"  Shape: {weekly_data.shape}")
print(f"  Index type: {type(weekly_data.index)}")
print(f"  Index dtype: {weekly_data.index.dtype}")
print(f"  First 3 index values: {list(weekly_data.index[:3])}")
print(f"  Last 3 index values: {list(weekly_data.index[-3:])}")

# Check day of week
first_date = weekly_data.index[0]
print(f"  First date: {first_date.date()} ({first_date.strftime('%A')})")

# Verify all dates are same day of week
days_of_week = weekly_data.index.dayofweek
unique_days = days_of_week.unique()
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
if len(unique_days) == 1:
    print(f"  ✓ All dates are {day_names[unique_days[0]]}s (consistent weekly data)")
else:
    print(f"  ⚠️  Mixed days of week: {[day_names[d] for d in unique_days]}")

print(f"\n  Date gaps analysis:")
date_diffs = pd.Series(weekly_data.index).diff().dt.days.dropna()
print(f"    Min gap: {date_diffs.min()} days")
print(f"    Max gap: {date_diffs.max()} days")
print(f"    Mean gap: {date_diffs.mean():.1f} days")

# Note: Weather and flu data will be loaded in cell 16
print(f"\n  Note: External data (weather, flu) will be loaded and diagnosed in the external data section.")
print("="*60)

DIAGNOSTIC: HOSPITAL DATA CHECK

📊 Hospital Weekly Data:
  Shape: (197, 16)
  Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
  Index dtype: datetime64[ns]
  First 3 index values: [Timestamp('2020-07-19 00:00:00'), Timestamp('2020-07-26 00:00:00'), Timestamp('2020-08-02 00:00:00')]
  Last 3 index values: [Timestamp('2024-04-07 00:00:00'), Timestamp('2024-04-14 00:00:00'), Timestamp('2024-04-21 00:00:00')]
  First date: 2020-07-19 (Sunday)
  ✓ All dates are Sundays (consistent weekly data)

  Date gaps analysis:
    Min gap: 7.0 days
    Max gap: 7.0 days
    Mean gap: 7.0 days

  Note: External data (weather, flu) will be loaded and diagnosed in the external data section.


In [ ]:
# ============================================================================
# DIAGNOSTIC: KPI DataFrame Check
# ============================================================================

print("="*60)
print("DIAGNOSTIC: KPI DATAFRAME CHECK")
print("="*60)

print(f"\n📊 KPI DataFrame:")
print(f"  Shape: {kpi_df.shape}")
print(f"  Columns: {list(kpi_df.columns)}")
print(f"  Index type: {type(kpi_df.index)}")
print(f"  Date range: {kpi_df.index.min().date()} to {kpi_df.index.max().date()}")

# Check for missing values
print(f"\n  Missing values:")
for col in kpi_df.columns:
    missing = kpi_df[col].isnull().sum()
    pct = missing / len(kpi_df) * 100
    if missing == 0:
        print(f"    {col}: ✓ Complete (0 missing)")
    else:
        print(f"    {col}: ⚠️  {missing} missing ({pct:.1f}%)")

# Basic statistics
print(f"\n  Value ranges:")
for col in kpi_df.columns:
    if kpi_df[col].dtype in [np.float64, np.int64]:
        print(f"    {col}: {kpi_df[col].min():.1f} to {kpi_df[col].max():.1f}")

print("="*60)

DIAGNOSTIC: KPI DATAFRAME CHECK

📊 KPI DataFrame:
  Shape: (197, 10)
  Columns: ['bed_occupancy', 'occupancy_rate', 'bed_capacity', 'icu_occupied', 'icu_capacity', 'icu_occupancy_rate', 'covid_inpatients', 'covid_icu', 'covid_patient_pct', 'staffed_beds']
  Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
  Date range: 2020-07-19 to 2024-04-21

  Missing values:
    bed_occupancy: ✓ Complete (0 missing)
    occupancy_rate: ✓ Complete (0 missing)
    bed_capacity: ✓ Complete (0 missing)
    icu_occupied: ⚠️  2 missing (1.0%)
    icu_capacity: ⚠️  2 missing (1.0%)
    icu_occupancy_rate: ⚠️  2 missing (1.0%)
    covid_inpatients: ⚠️  46 missing (23.4%)
    covid_icu: ⚠️  8 missing (4.1%)
    covid_patient_pct: ⚠️  46 missing (23.4%)
    staffed_beds: ✓ Complete (0 missing)

  Value ranges:
    bed_occupancy: 1882.0 to 2510.7
    occupancy_rate: 66.8 to 98.4
    bed_capacity: 2271.0 to 2895.9
    icu_occupied: 225.7 to 340.0
    icu_capacity: 297.0 to 447.7
    icu_occupa

In [ ]:
# ============================================================================
# NOTE: Robust merge function is now integrated into merge_external_data()
# in the Helper Functions section above. This cell is kept for reference.
# ============================================================================

print("Note: The merge_external_data() function in the Helper Functions section")
print("now includes robust date alignment with the following features:")
print("  • Day-of-week verification")
print("  • Exact match detection")
print("  • Nearest-date matching within 7 days for misaligned dates")
print("  • Detailed diagnostics for debugging alignment issues")
print("")
print("The merge will be executed in the next section.")

Note: The merge_external_data() function in the Helper Functions section
now includes robust date alignment with the following features:
  • Day-of-week verification
  • Exact match detection
  • Nearest-date matching within 7 days for misaligned dates
  • Detailed diagnostics for debugging alignment issues

The merge will be executed in the next section.


## Process External Data (Weather & Flu)

Load and merge external data sources (weather, flu) with the weekly bed occupancy data.

**Note:** External data from `01a_data_extraction.ipynb` is already aligned to hospital dates (matching hospital date range). This notebook simply loads and merges the pre-aligned data using a simple date-index merge (no nearest-date matching needed).


In [ ]:
# Process external data
print("="*60)
print("PROCESSING EXTERNAL DATA (Weather & Flu)")
print("="*60)

# Find external data directory
external_dirs = [DRIVE_DATA_EXTERNAL, DATA_EXTERNAL]
external_dirs = [d for d in external_dirs if d and os.path.exists(d)]

external_data_dir = external_dirs[0] if external_dirs else None

weather_df = None
flu_df = None
weekly_with_external = None

if external_data_dir:
    print(f"\n📂 External data directory: {external_data_dir}")

    # Load external data
    print(f"\n📥 Loading external data...")
    weather_df, flu_df = load_external_data(external_data_dir)

    # Display weather data info and validate alignment
    if weather_df is not None and len(weather_df) > 0:
        print(f"\n🌤️  Weather Data:")
        print(f"  Records: {len(weather_df)}")
        print(f"  Date range: {weather_df.index.min().date()} to {weather_df.index.max().date()}")
        print(f"  Columns: {list(weather_df.columns)}")

        # Validate date alignment with hospital data
        hospital_start = weekly_data.index.min()
        hospital_end = weekly_data.index.max()
        weather_start = weather_df.index.min()
        weather_end = weather_df.index.max()

        if len(weather_df) != len(weekly_data):
            print(f"  ⚠️  Warning: Weather records ({len(weather_df)}) don't match hospital weeks ({len(weekly_data)})")
            print(f"     Expected: {len(weekly_data)} records (01a should have aligned this)")
        elif weather_start != hospital_start or weather_end != hospital_end:
            print(f"  ⚠️  Warning: Weather date range doesn't match hospital date range")
            print(f"     Hospital: {hospital_start.date()} to {hospital_end.date()}")
            print(f"     Weather: {weather_start.date()} to {weather_end.date()}")
        else:
            print(f"  ✓ Date alignment validated: Matches hospital date range")

    # Display flu data info and validate alignment
    if flu_df is not None and len(flu_df) > 0:
        print(f"\n🦠 Flu Data:")
        print(f"  Records: {len(flu_df)}")
        print(f"  Date range: {flu_df.index.min().date()} to {flu_df.index.max().date()}")
        print(f"  Columns: {list(flu_df.columns)}")

        # Validate date alignment with hospital data
        hospital_start = weekly_data.index.min()
        hospital_end = weekly_data.index.max()
        flu_start = flu_df.index.min()
        flu_end = flu_df.index.max()

        if len(flu_df) != len(weekly_data):
            print(f"  ⚠️  Warning: Flu records ({len(flu_df)}) don't match hospital weeks ({len(weekly_data)})")
            print(f"     Expected: {len(weekly_data)} records (01a should have aligned this)")
        elif flu_start != hospital_start or flu_end != hospital_end:
            print(f"  ⚠️  Warning: Flu date range doesn't match hospital date range")
            print(f"     Hospital: {hospital_start.date()} to {hospital_end.date()}")
            print(f"     Flu: {flu_start.date()} to {flu_end.date()}")
        else:
            print(f"  ✓ Date alignment validated: Matches hospital date range")

    # Merge external data with weekly bed occupancy (simple merge - dates already aligned in 01a)
    if weather_df is not None or flu_df is not None:
        print(f"\n📦 Merging external data with weekly bed occupancy...")
        print(f"  Note: Using simple date-index merge (data pre-aligned in 01a_data_extraction.ipynb)")
        weekly_with_external = merge_external_data(weekly_data, weather_df, flu_df)

        print(f"\n✓ Merged dataset:")
        print(f"  Shape: {weekly_with_external.shape}")
        print(f"  Columns: {list(weekly_with_external.columns)}")

        # Check for missing values after merge
        missing_after_merge = weekly_with_external.isnull().sum()
        cols_with_missing = missing_after_merge[missing_after_merge > 0]
        if len(cols_with_missing) > 0:
            print(f"\n  Missing values after merge:")
            for col, count in cols_with_missing.items():
                pct = count / len(weekly_with_external) * 100
                print(f"    {col}: {count} ({pct:.1f}%)")
        else:
            print(f"  ✓ No missing values after merge")
    else:
        print(f"\n⚠️  No external data to merge")
        weekly_with_external = weekly_data.copy()
else:
    print(f"\n⚠️  External data directory not found")
    print(f"   Expected: {DATA_EXTERNAL}")
    print(f"   External data is optional - processing will continue without it")
    weekly_with_external = weekly_data.copy()

PROCESSING EXTERNAL DATA (Weather & Flu)

📂 External data directory: /content/drive/MyDrive/hospital_occupancy_forecasting/data/external

📥 Loading external data...
  ✓ Loaded weather data: 197 records
    Date range: 2020-07-19 to 2024-04-21
    First date: 2020-07-19 (Sunday)
    Data completeness: 100.0% non-null values
  ✓ Loaded flu data: 197 records
    Date range: 2020-07-19 to 2024-04-21
    First date: 2020-07-19 (Sunday)
    flu_activity_level: 197/197 non-null (100.0%)

🌤️  Weather Data:
  Records: 197
  Date range: 2020-07-19 to 2024-04-21
  Columns: ['temp_avg', 'temp_max', 'temp_min', 'precipitation', 'wind_speed', 'pressure']
  ✓ Date alignment validated: Matches hospital date range

🦠 Flu Data:
  Records: 197
  Date range: 2020-07-19 to 2024-04-21
  Columns: ['epiweek', 'flu_activity_level', 'wili', 'ili', 'is_flu_season', 'region', 'is_proxy']
  ✓ Date alignment validated: Matches hospital date range

📦 Merging external data with weekly bed occupancy...
  Note: Using s

## Save Processed Data

Save all processed data files for use in downstream notebooks.


In [ ]:
# Save processed data
print("="*60)
print("SAVING PROCESSED DATA")
print("="*60)

saved_files = []

# 1. Save weekly occupancy time series
print(f"\n📁 Saving weekly occupancy...")
weekly_occupancy_file = os.path.join(DATA_PROCESSED, 'weekly_occupancy.csv')
weekly_data.to_csv(weekly_occupancy_file)
saved_files.append(('weekly_occupancy.csv', len(weekly_data), list(weekly_data.columns)))
print(f"  ✓ Saved: {weekly_occupancy_file}")

if DRIVE_DATA_PROCESSED:
    weekly_occupancy_file_drive = os.path.join(DRIVE_DATA_PROCESSED, 'weekly_occupancy.csv')
    weekly_data.to_csv(weekly_occupancy_file_drive)
    print(f"  ✓ Also saved to Google Drive")

# 2. Save KPI dataframe
print(f"\n📁 Saving weekly KPIs...")
kpi_file = os.path.join(DATA_PROCESSED, 'weekly_kpis.csv')
kpi_df.to_csv(kpi_file)
saved_files.append(('weekly_kpis.csv', len(kpi_df), list(kpi_df.columns)))
print(f"  ✓ Saved: {kpi_file}")

if DRIVE_DATA_PROCESSED:
    kpi_file_drive = os.path.join(DRIVE_DATA_PROCESSED, 'weekly_kpis.csv')
    kpi_df.to_csv(kpi_file_drive)
    print(f"  ✓ Also saved to Google Drive")

# 3. Save combined dataset with external data
if weekly_with_external is not None:
    print(f"\n📁 Saving weekly occupancy with external data...")
    combined_file = os.path.join(DATA_PROCESSED, 'weekly_occupancy_with_external.csv')
    weekly_with_external.to_csv(combined_file)
    saved_files.append(('weekly_occupancy_with_external.csv', len(weekly_with_external), list(weekly_with_external.columns)))
    print(f"  ✓ Saved: {combined_file}")

    if DRIVE_DATA_PROCESSED:
        combined_file_drive = os.path.join(DRIVE_DATA_PROCESSED, 'weekly_occupancy_with_external.csv')
        weekly_with_external.to_csv(combined_file_drive)
        print(f"  ✓ Also saved to Google Drive")

# 4. Create simplified bed occupancy file (target variable only)
print(f"\n📁 Creating simplified bed occupancy file...")
# This file contains just the target variables for easy model loading
bed_occ_file = os.path.join(DATA_PROCESSED, 'bed_occupancy.csv')
bed_occ_dict = {'bed_occupancy': weekly_data['bed_occupancy']}
if 'occupancy_rate' in weekly_data.columns:
    bed_occ_dict['occupancy_rate'] = weekly_data['occupancy_rate']
bed_occ_df = pd.DataFrame(bed_occ_dict, index=weekly_data.index)
bed_occ_df.to_csv(bed_occ_file)
saved_files.append(('bed_occupancy.csv', len(bed_occ_df), list(bed_occ_df.columns)))
print(f"  ✓ Saved: {bed_occ_file}")

if DRIVE_DATA_PROCESSED:
    bed_occ_file_drive = os.path.join(DRIVE_DATA_PROCESSED, 'bed_occupancy.csv')
    bed_occ_df.to_csv(bed_occ_file_drive)

# Summary
print(f"\n" + "="*60)
print(f"📋 SAVED FILES SUMMARY")
print("="*60)
for filename, rows, cols in saved_files:
    print(f"  {filename}: {rows} rows × {len(cols)} columns")

SAVING PROCESSED DATA

📁 Saving weekly occupancy...
  ✓ Saved: /content/hospital_occupancy_forecasting/data/processed/weekly_occupancy.csv
  ✓ Also saved to Google Drive

📁 Saving weekly KPIs...
  ✓ Saved: /content/hospital_occupancy_forecasting/data/processed/weekly_kpis.csv
  ✓ Also saved to Google Drive

📁 Saving weekly occupancy with external data...
  ✓ Saved: /content/hospital_occupancy_forecasting/data/processed/weekly_occupancy_with_external.csv
  ✓ Also saved to Google Drive

📁 Creating simplified bed occupancy file...
  ✓ Saved: /content/hospital_occupancy_forecasting/data/processed/bed_occupancy.csv

📋 SAVED FILES SUMMARY
  weekly_occupancy.csv: 197 rows × 16 columns
  weekly_kpis.csv: 197 rows × 10 columns
  weekly_occupancy_with_external.csv: 197 rows × 27 columns
  bed_occupancy.csv: 197 rows × 2 columns


## Create Summary Statistics

Create a summary statistics JSON file for documentation and quick reference.


In [ ]:
# Create summary statistics file
print("="*60)
print("CREATING SUMMARY STATISTICS")
print("="*60)

summary_stats = {
    'data_source': 'COVID-19 Reported Patient Impact and Hospital Capacity by Facility',
    'hospital': 'ADVENTHEALTH ORLANDO',
    'location': 'Orlando, FL',
    'frequency': 'weekly',
    'processing_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}

# Weekly occupancy statistics
summary_stats['weekly_occupancy'] = {
    'weeks': int(len(weekly_data)),
    'date_range_start': str(weekly_data.index.min().date()),
    'date_range_end': str(weekly_data.index.max().date()),
    'columns': list(weekly_data.columns)
}

# Bed occupancy statistics
if 'bed_occupancy' in weekly_data.columns:
    occ = weekly_data['bed_occupancy'].dropna()
    summary_stats['bed_occupancy'] = {
        'mean': float(occ.mean()),
        'std': float(occ.std()),
        'min': float(occ.min()),
        'max': float(occ.max()),
        'median': float(occ.median())
    }

# Occupancy rate statistics
if 'occupancy_rate' in weekly_data.columns:
    rate = weekly_data['occupancy_rate'].dropna()
    summary_stats['occupancy_rate'] = {
        'mean': float(rate.mean()),
        'std': float(rate.std()),
        'min': float(rate.min()),
        'max': float(rate.max()),
        'median': float(rate.median())
    }

# External data coverage
if weather_df is not None and len(weather_df) > 0:
    summary_stats['weather_data'] = {
        'records': int(len(weather_df)),
        'date_range_start': str(weather_df.index.min().date()),
        'date_range_end': str(weather_df.index.max().date()),
        'columns': list(weather_df.columns)
    }

if flu_df is not None and len(flu_df) > 0:
    summary_stats['flu_data'] = {
        'records': int(len(flu_df)),
        'date_range_start': str(flu_df.index.min().date()),
        'date_range_end': str(flu_df.index.max().date()),
        'columns': list(flu_df.columns)
    }

# Imputation info
if 'is_imputed' in weekly_data.columns:
    imputed_count = int(weekly_data['is_imputed'].sum())
    summary_stats['imputation'] = {
        'imputed_weeks': imputed_count,
        'imputed_pct': float(imputed_count / len(weekly_data) * 100),
        'method': 'time-aware linear interpolation'
    }

# Save summary statistics
summary_file = os.path.join(DATA_PROCESSED, 'preprocessing_summary.json')
with open(summary_file, 'w') as f:
    json.dump(summary_stats, f, indent=2)
print(f"\n✓ Summary statistics saved: {summary_file}")

if DRIVE_DATA_PROCESSED:
    summary_file_drive = os.path.join(DRIVE_DATA_PROCESSED, 'preprocessing_summary.json')
    with open(summary_file_drive, 'w') as f:
        json.dump(summary_stats, f, indent=2)
    print(f"✓ Also saved to Google Drive")

# Display summary
print(f"\n📊 Summary Statistics:")
print(json.dumps(summary_stats, indent=2))

CREATING SUMMARY STATISTICS

✓ Summary statistics saved: /content/hospital_occupancy_forecasting/data/processed/preprocessing_summary.json
✓ Also saved to Google Drive

📊 Summary Statistics:
{
  "data_source": "COVID-19 Reported Patient Impact and Hospital Capacity by Facility",
  "hospital": "ADVENTHEALTH ORLANDO",
  "location": "Orlando, FL",
  "frequency": "weekly",
  "processing_date": "2025-12-18 06:10:59",
  "weekly_occupancy": {
    "weeks": 197,
    "date_range_start": "2020-07-19",
    "date_range_end": "2024-04-21",
    "columns": [
      "aligned_date",
      "bed_occupancy",
      "bed_capacity",
      "occupancy_rate",
      "coverage",
      "hospital_name",
      "city",
      "state",
      "is_imputed",
      "icu_occupied",
      "icu_capacity",
      "covid_inpatients",
      "covid_icu",
      "staffed_beds",
      "icu_occupancy_rate",
      "covid_patient_pct"
    ]
  },
  "bed_occupancy": {
    "mean": 2254.3670050761425,
    "std": 136.46660157179886,
    "min":

## Final Verification

Verify all output files were created successfully.


In [ ]:
# Final verification
print("="*60)
print("FINAL VERIFICATION")
print("="*60)

expected_files = [
    'weekly_occupancy.csv',
    'weekly_kpis.csv',
    'weekly_occupancy_with_external.csv',
    'bed_occupancy.csv',  # simplified target variable file
    'preprocessing_summary.json'
]

print(f"\n📁 Checking output files in {DATA_PROCESSED}:")
all_exist = True
for filename in expected_files:
    filepath = os.path.join(DATA_PROCESSED, filename)
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  ✓ {filename} ({size_kb:.1f} KB)")
    else:
        print(f"  ❌ {filename} - NOT FOUND")
        all_exist = False

# Verify data can be loaded
print(f"\n📊 Loading verification:")
try:
    verify_df = pd.read_csv(os.path.join(DATA_PROCESSED, 'weekly_occupancy.csv'), index_col=0, parse_dates=True)
    print(f"  ✓ weekly_occupancy.csv loads successfully: {len(verify_df)} rows")
except Exception as e:
    print(f"  ❌ Error loading weekly_occupancy.csv: {e}")
    all_exist = False

# ============================================================
# FINAL DATE ALIGNMENT VERIFICATION
# ============================================================
print(f"\n" + "="*60)
print("FINAL DATE ALIGNMENT VERIFICATION")
print("="*60)

# Check the merged data for alignment issues
if weekly_with_external is not None:
    print(f"\n📊 Merged Dataset Analysis:")
    print(f"   Total records: {len(weekly_with_external)}")
    print(f"   Date range: {weekly_with_external.index.min().date()} to {weekly_with_external.index.max().date()}")
    print(f"   First date day: {weekly_with_external.index[0].strftime('%A')}")

    # Check each data category
    hospital_cols = ['bed_occupancy', 'bed_capacity', 'occupancy_rate']
    hospital_cols = [c for c in hospital_cols if c in weekly_with_external.columns]
    if hospital_cols:
        hospital_missing = weekly_with_external[hospital_cols].isnull().sum().sum()
        print(f"\n   Hospital data ({len(hospital_cols)} cols): {hospital_missing} missing values")

    weather_cols = ['temp_avg', 'temp_max', 'temp_min', 'precipitation']
    weather_cols = [c for c in weather_cols if c in weekly_with_external.columns]
    if weather_cols:
        weather_missing = weekly_with_external[weather_cols].isnull().sum().sum()
        weather_total = len(weekly_with_external) * len(weather_cols)
        weather_pct = weather_missing / weather_total * 100 if weather_total > 0 else 0
        if weather_missing == 0:
            print(f"   Weather data ({len(weather_cols)} cols): ✓ Complete (0 missing)")
        else:
            print(f"   Weather data ({len(weather_cols)} cols): ⚠️  {weather_missing} missing ({weather_pct:.1f}%)")

    flu_cols = ['flu_activity_level', 'is_flu_season']
    flu_cols = [c for c in flu_cols if c in weekly_with_external.columns]
    if flu_cols:
        flu_missing = weekly_with_external[flu_cols].isnull().sum().sum()
        flu_total = len(weekly_with_external) * len(flu_cols)
        flu_pct = flu_missing / flu_total * 100 if flu_total > 0 else 0
        if flu_missing == 0:
            print(f"   Flu data ({len(flu_cols)} cols): ✓ Complete (0 missing)")
        else:
            print(f"   Flu data ({len(flu_cols)} cols): ⚠️  {flu_missing} missing ({flu_pct:.1f}%)")

    # Show sample of merged data
    print(f"\n📋 Sample of Merged Data (first 3 rows):")
    sample_cols = ['bed_occupancy']
    if 'temp_avg' in weekly_with_external.columns:
        sample_cols.append('temp_avg')
    if 'flu_activity_level' in weekly_with_external.columns:
        sample_cols.append('flu_activity_level')
    print(weekly_with_external[sample_cols].head(3).to_string())

# Summary
print(f"\n" + "="*60)
if all_exist:
    print("✅ ALL VERIFICATION CHECKS PASSED")
    print("\n📋 Preprocessing complete! Output files:")
    print(f"  • weekly_occupancy.csv - Weekly bed occupancy time series (all columns)")
    print(f"  • weekly_kpis.csv - KPIs (bed_occupancy, occupancy_rate, ICU, ventilator, COVID metrics if available)")
    print(f"  • weekly_occupancy_with_external.csv - Combined with weather/flu")
    print(f"  • bed_occupancy.csv - Simplified target variable file")
    print(f"  • preprocessing_summary.json - Summary statistics")
else:
    print("⚠️  SOME VERIFICATION CHECKS FAILED")
    print("   Please review the output above")

print("="*60)

FINAL VERIFICATION

📁 Checking output files in /content/hospital_occupancy_forecasting/data/processed:
  ✓ weekly_occupancy.csv (28.9 KB)
  ✓ weekly_kpis.csv (19.8 KB)
  ✓ weekly_occupancy_with_external.csv (51.7 KB)
  ✓ bed_occupancy.csv (6.9 KB)
  ✓ preprocessing_summary.json (1.6 KB)

📊 Loading verification:
  ✓ weekly_occupancy.csv loads successfully: 197 rows

FINAL DATE ALIGNMENT VERIFICATION

📊 Merged Dataset Analysis:
   Total records: 197
   Date range: 2020-07-19 to 2024-04-21
   First date day: Sunday

   Hospital data (3 cols): 0 missing values
   Weather data (4 cols): ✓ Complete (0 missing)
   Flu data (2 cols): ✓ Complete (0 missing)

📋 Sample of Merged Data (first 3 rows):
            bed_occupancy   temp_avg  flu_activity_level
date                                                    
2020-07-19         2011.0  84.714286                 0.5
2020-07-26         2022.9  82.888571                 0.5
2020-08-02         2007.7  83.145714                 0.5

✅ ALL VERIFICATI

## Summary

This notebook processed weekly bed occupancy data from the COVID-19 Hospital Capacity dataset:

**Data Source:**
- COVID-19 Reported Patient Impact and Hospital Capacity by Facility
- Hospital: ADVENTHEALTH ORLANDO, Orlando, FL
- Date Range: 2020-07-19 to 2024-04-21 (197 weeks with valid occupancy data)

**Processing Steps:**
1. ✅ Loaded weekly bed occupancy data from `01a_data_extraction.ipynb`
2. ✅ Validated weekly frequency (7-day intervals)
3. ✅ Performed data quality checks
4. ✅ Created KPI DataFrame (bed_occupancy, occupancy_rate, and additional metrics: ICU, ventilator, COVID patients if available)
5. ✅ Merged external data (weather, flu) if available
6. ✅ Saved all processed files

**Output Files:**
- `weekly_occupancy.csv` - Weekly bed occupancy time series (all columns)
- `weekly_kpis.csv` - Weekly KPIs for modeling (bed_occupancy, occupancy_rate, bed_capacity, and additional metrics: ICU, ventilator, COVID patients if available)
- `weekly_occupancy_with_external.csv` - Combined dataset with weather and flu features
- `bed_occupancy.csv` - Simplified target variable file (bed_occupancy, occupancy_rate)
- `preprocessing_summary.json` - Summary statistics and data quality metrics

**Next Steps:**
- Proceed to `02_eda.ipynb` for exploratory data analysis
- Or proceed to `03_feature_engineering.ipynb` to create features for modeling
